## Before you start — read this (Kaggle)

**What this notebook does**  
Runs the **full cross-subject EEG decoding** pipeline on **Kaggle** (free GPU, more RAM than Colab). It loads your **preprocessed** .fif files from a Kaggle dataset, trains a Transformer with Leave-One-Subject-Out, and writes results to /kaggle/working/vibci_outputs/. You download outputs from the run's **Output** tab.

**What you need on Kaggle**  
Only the **preprocessed epoch files** from your computer. These are the `.fif` files inside your project’s `outputs/preprocessed/` folder (e.g. `sub-01_ses-01_AVI_clean-epo.fif`). Do **not** upload the raw `.bdf` files (use only preprocessed .fif from outputs/preprocessed/; enable GPU in Settings).

**Dataset layout**  
Upload so **all .fif files are in the dataset root** (no subfolders). Names: sub-01_ses-01_AVI_clean-epo.fif, etc.  

**In cell 1** set KAGGLE_DATASET_SLUG to your dataset slug (e.g. yourusername/vibci-preprocessed).  

**Settings**  
- **QUICK_MODE = False** → full 22 subjects, 22 folds.  
- **USE_EA = True** (in cell 5b) → Euclidean Alignment is applied.  
- **MAX_EPOCHS = 100**, early stopping patience 15, 10-epoch LR warmup, gradient clipping 1.0.  

Run all cells in order. When done, open the run's **Output** tab on Kaggle and download `vibci_outputs.zip` and/or the `vibci_outputs/` folder.

## 1. Set paths

Set `KAGGLE_DATASET_SLUG`; data is read from `/kaggle/input/<slug>/`, outputs to `/kaggle/working/`.

## 1. Set paths (Kaggle)

Data is read from the Kaggle input dataset you attach; outputs are written to `/kaggle/working/`. Set `KAGGLE_DATASET_SLUG` to your dataset's slug (e.g. `yourusername/vibci-preprocessed`).

In [ ]:
import os

# Your Kaggle dataset slug (username/dataset-name). Data will be at /kaggle/input/<slug>/
KAGGLE_DATASET_SLUG = 'datasets/youssof20/vi-bci-preprocessed-epochs'  # path under /kaggle/input/

if os.path.exists('/kaggle/working'):
    DATASET_PATH = f'/kaggle/input/{KAGGLE_DATASET_SLUG.strip().replace(" ", "-")}/'
    OUTPUTS_PATH = '/kaggle/working/vibci_outputs/'
else:
    DATASET_PATH = f'/kaggle/input/{KAGGLE_DATASET_SLUG}/'  # fallback for local test
    OUTPUTS_PATH = './vibci_outputs/'

print('DATASET_PATH:', DATASET_PATH)
print('OUTPUTS_PATH:', OUTPUTS_PATH)

## 2. Install dependencies

Installs MNE-Python (EEG), PyTorch, and scikit-learn. Re-run this cell if you get import errors.

In [ ]:
!pip install mne torch torchvision scikit-learn tqdm -q

## 3. Verify GPU

Confirm Kaggle is using a GPU. If this shows CPU, go to **Settings** (right sidebar) → **Accelerator** → **GPU** and save.

In [ ]:
import torch
print(f"GPU available: {torch.cuda.is_available()}")
print(f"Device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

## 4. Data loader (lazy loading)

Does **not** load all subjects into RAM. Discovers which subjects have data via file scan; each subject is loaded one at a time inside the LOSO loop (cell 7), converted to arrays, then freed with `del` and `gc.collect()` to keep peak memory ~4GB instead of ~20GB.

In [ ]:
import os
import gc
import numpy as np
import pandas as pd
import mne
import warnings
warnings.filterwarnings("ignore")

QUICK_MODE = False  # True: 5 subjects, 3 folds. False: 22 subjects, 22 folds

SUBJECT_IDS = [f"{i:02d}" for i in range(1, 23)]
SESSIONS = ("01", "02")
TASKS = ("AVI", "FVI", "OVI")
TASK_TO_SUPER = {"AVI": "LIVING", "FVI": "GEOMETRIC", "OVI": "OBJECT"}
TASK_VALUE_TO_LABEL10 = {
    ("AVI", 1): 0, ("AVI", 2): 1, ("AVI", 3): 2,
    ("FVI", 1): 5, ("FVI", 2): 4, ("FVI", 3): 3,
    ("OVI", 1): 6, ("OVI", 2): 7, ("OVI", 3): 8, ("OVI", 4): 9,
}
SUBJECTS_SESSION_1_ONLY = ("09", "10")
SKIP_RUNS = {("08", "02", "AVI")}

def get_subject_ids_with_data(dataset_root, quick_mode=False):
    """Return list of subject IDs that have at least one valid .fif file (no loading)."""
    sub_ids = SUBJECT_IDS[:5] if quick_mode else SUBJECT_IDS
    out = []
    for sub_id in sub_ids:
        for ses in SESSIONS:
            if sub_id in SUBJECTS_SESSION_1_ONLY and ses == "02":
                continue
            for task in TASKS:
                if (sub_id, ses, task) in SKIP_RUNS:
                    continue
                fpath = os.path.join(dataset_root, f"sub-{sub_id}_ses-{ses}_{task}_clean-epo.fif")
                if os.path.isfile(fpath):
                    out.append(sub_id)
                    break
            else:
                continue
            break
    return out

def load_one_subject(sub_id, dataset_root, skipped_log):
    """Load one subject's epochs. Returns mne.Epochs or None. Caller must del epochs; gc.collect() after use."""
    list_ep = []
    for ses in SESSIONS:
        if sub_id in SUBJECTS_SESSION_1_ONLY and ses == "02":
            continue
        for task in TASKS:
            if (sub_id, ses, task) in SKIP_RUNS:
                skipped_log.append(f"sub-{sub_id} ses-{ses} {task} (known missing)")
                continue
            fpath = os.path.join(dataset_root, f"sub-{sub_id}_ses-{ses}_{task}_clean-epo.fif")
            if not os.path.isfile(fpath):
                skipped_log.append(f"sub-{sub_id} ses-{ses} {task}: file not found")
                continue
            ep = mne.read_epochs(fpath, verbose=False)
            n = len(ep)
            if n == 0:
                skipped_log.append(f"sub-{sub_id} ses-{ses} {task}: no epochs")
                continue
            events = ep.events
            label_10 = np.array([TASK_VALUE_TO_LABEL10.get((task, int(ev[2])), -1) for ev in events])
            if np.any(label_10 < 0):
                skipped_log.append(f"sub-{sub_id} ses-{ses} {task}: unknown event value")
                continue
            label_3 = (label_10 // 3).clip(0, 2)
            supercat = TASK_TO_SUPER[task]
            ep.metadata = pd.DataFrame({
                "supercategory": [supercat] * n,
                "label_10": label_10,
                "label_3": label_3,
                "task": [task] * n,
            }, index=range(n))
            list_ep.append(ep)
    if not list_ep:
        skipped_log.append(f"sub-{sub_id}: no data at all")
        return None
    return mne.concatenate_epochs(list_ep, on_mismatch="ignore", verbose=False)

skipped = []
all_loaded_ids = get_subject_ids_with_data(DATASET_PATH, quick_mode=QUICK_MODE)
fold_test_ids = all_loaded_ids[:3] if QUICK_MODE else all_loaded_ids
n_folds = len(fold_test_ids)
print(f"Subjects with data: {len(all_loaded_ids)}. Folds: {n_folds} (test subjects: {fold_test_ids})")

## 5. Feature extraction

Crop 0–3 s, downsample to 250 Hz (750 timepoints), z-score per channel on training set only. Tensors are moved to GPU in the training loop.

In [ ]:
CROP_TMIN, CROP_TMAX = 0.0, 3.0
TARGET_SFREQ = 250
EXPECTED_N_TIMES = 750
N_CHANNELS = 32

def epochs_to_arrays(epochs):
    ep = epochs.copy()
    ep.crop(tmin=CROP_TMIN, tmax=CROP_TMAX)
    ep.resample(TARGET_SFREQ, verbose=False)
    X = ep.get_data(picks="eeg")
    n_times = X.shape[2]
    if n_times != EXPECTED_N_TIMES:
        from scipy.signal import resample
        X = resample(X, EXPECTED_N_TIMES, axis=2)
    y10 = epochs.metadata["label_10"].values.astype(np.int64)
    y3 = epochs.metadata["label_3"].values.astype(np.int64)
    return X, y10, y3

def zscore_fit_transform(X_train, X_apply):
    # Per-channel normalization over trials and time (shape 1, 32, 1)
    mean = X_train.mean(axis=(0, 2), keepdims=True)
    std = X_train.std(axis=(0, 2), keepdims=True)
    std[std < 1e-8] = 1.0
    return (X_train - mean) / std, (X_apply - mean) / std

# Get channel names for topomap later (load one subject briefly, then free)
epochs_ref = load_one_subject(all_loaded_ids[0], DATASET_PATH, [])
if epochs_ref is not None:
    try:
        montage = mne.channels.make_standard_montage("standard_1020")
        epochs_ref.set_montage(montage, on_missing="ignore", verbose=False)
    except Exception:
        pass
    ch_names = epochs_ref.ch_names[:32] if len(epochs_ref.ch_names) >= 32 else epochs_ref.ch_names
    del epochs_ref
    gc.collect()
else:
    ch_names = []
print(f"Feature shape per subject: (n_epochs, {N_CHANNELS}, {EXPECTED_N_TIMES})")

## 5b. Euclidean Alignment (optional)

Reduces inter-subject domain shift by aligning each subject's covariance to a common space. Set **USE_EA = True** for the full run (recommended).

In [ ]:
from scipy.linalg import fractional_matrix_power

USE_EA = True  # True: apply Euclidean Alignment per subject (recommended for cross-subject)

def euclidean_align(X, reg=1e-5):
    n_trials, n_channels, n_times = X.shape
    T = n_times
    R_list = []
    for i in range(n_trials):
        x_i = X[i]
        R_i = (x_i @ x_i.T) / T
        R_list.append(R_i)
    R_bar = np.mean(R_list, axis=0) + reg * np.eye(n_channels)
    R_bar_inv_sqrt = fractional_matrix_power(R_bar, -0.5)
    X_aligned = np.zeros_like(X)
    for i in range(n_trials):
        X_aligned[i] = R_bar_inv_sqrt @ X[i]
    return X_aligned

if USE_EA:
    print("Euclidean Alignment will be applied per subject when loading in the LOSO loop.")
else:
    print("EA skipped (USE_EA=False).")

## 6. Model: EEGTransformer

Same architecture as local: spatial conv → temporal conv → Transformer (2 layers, 4 heads) → classifier. Instantiated with `.to(device)` so it runs on GPU.

In [ ]:
import math
import torch.nn as nn

class EEGTransformer(nn.Module):
    def __init__(self, n_classes=10, n_channels=32, n_times=750, dropout=0.25, num_layers=2, nhead=4):
        super().__init__()
        self.n_channels = n_channels
        self.n_times = n_times
        self.spatial_conv = nn.Conv2d(1, 32, kernel_size=(n_channels, 1))
        self.bn1 = nn.BatchNorm2d(32)
        self.drop1 = nn.Dropout2d(dropout)
        self.temporal_conv = nn.Conv2d(32, 64, kernel_size=(1, 16), stride=(1, 4))
        self.bn2 = nn.BatchNorm2d(64)
        self.drop2 = nn.Dropout2d(dropout)
        self.d_model = 64
        self.pos_scale = 1.0
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=64, nhead=nhead, dim_feedforward=128, dropout=dropout,
            activation="gelu", batch_first=True, norm_first=False,
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.drop3 = nn.Dropout(dropout)
        self.pool = nn.AdaptiveAvgPool1d(1)
        self.fc1 = nn.Linear(64, 64)
        self.drop4 = nn.Dropout(0.5)
        self.fc2 = nn.Linear(64, n_classes)

    def _sinusoidal_encoding(self, seq_len, d_model, device):
        pe = torch.zeros(seq_len, d_model, device=device)
        for i in range(seq_len):
            for j in range(0, d_model, 2):
                pe[i, j] = math.sin(i / 10000 ** (j / d_model))
                if j + 1 < d_model:
                    pe[i, j + 1] = math.cos(i / 10000 ** (j / d_model))
        return pe * self.pos_scale

    def forward(self, x):
        x = self.spatial_conv(x)
        x = self.bn1(x)
        x = torch.nn.functional.elu(x)
        x = self.drop1(x)
        x = self.temporal_conv(x)
        x = self.bn2(x)
        x = torch.nn.functional.elu(x)
        x = self.drop2(x)
        x = x.squeeze(2).permute(0, 2, 1)
        seq_len = x.size(1)
        pe = self._sinusoidal_encoding(seq_len, self.d_model, x.device)
        x = x + pe.unsqueeze(0)
        x = self.transformer(x)
        x = self.drop3(x)
        x = x.permute(0, 2, 1)
        x = self.pool(x).squeeze(-1)
        x = self.fc1(x)
        x = torch.nn.functional.elu(x)
        x = self.drop4(x)
        x = self.fc2(x)
        return x

    def get_spatial_conv_weights(self):
        w = self.spatial_conv.weight
        return w.squeeze(-1).squeeze(1)

def get_spatial_weights_for_topomap(model):
    w = model.get_spatial_conv_weights()
    if w is None:
        return None
    return w.abs().sum(dim=0).detach().cpu().numpy()

N_CLASSES_10 = 10
MAX_EPOCHS = 100
BATCH_SIZE = 32
LR = 1e-3
WEIGHT_DECAY = 1e-4
EARLY_STOP_PATIENCE = 15
WARMUP_EPOCHS = 10
VAL_FRAC_SUBJECTS = 0.1
CHANCE_10 = 10.0
CHANCE_3 = 100.0 / 3.0
CLASS_NAMES_10 = ["dog", "bird", "fish", "circle", "square", "pentagram", "scissor", "watch", "cup", "chair"]
FLAG_SUBJECT = "19"

## 7. LOSO training loop

Leave-one-subject-out: each fold trains on all but one subject, tests on the held-out subject. Checkpoints, logs, and figures go to `OUTPUTS_PATH`. Progress bars show fold and epoch.

In [ ]:
from torch.utils.data import TensorDataset, DataLoader, WeightedRandomSampler
from tqdm.notebook import tqdm

os.makedirs(os.path.join(OUTPUTS_PATH, 'models'), exist_ok=True)
os.makedirs(os.path.join(OUTPUTS_PATH, 'logs'), exist_ok=True)
os.makedirs(os.path.join(OUTPUTS_PATH, 'figures'), exist_ok=True)

all_y10_true, all_y10_pred = [], []
all_y3_true, all_y3_pred = [], []
fold_results = []
best_fold_sub_id = None
best_fold_acc_3 = -1.0
best_fold_state = None
best_fold_model = None

for fold_idx, test_sub_id in enumerate(tqdm(fold_test_ids, desc="Folds")):
    tqdm.write(f"Fold {fold_idx+1}/{n_folds} - Testing on sub-{test_sub_id}")
    train_sub_ids = [s for s in all_loaded_ids if s != test_sub_id]
    n_val = max(1, int(len(train_sub_ids) * VAL_FRAC_SUBJECTS))
    val_sub_ids = train_sub_ids[-n_val:]
    train_sub_ids = train_sub_ids[:-n_val]

    X_train_list, y10_train_list, y3_train_list = [], [], []
    for s in train_sub_ids:
        epochs = load_one_subject(s, DATASET_PATH, skipped)
        if epochs is None:
            continue
        X, y10, y3 = epochs_to_arrays(epochs)
        del epochs
        gc.collect()
        if USE_EA:
            X = euclidean_align(X)
        X_train_list.append(X)
        y10_train_list.append(y10)
        y3_train_list.append(y3)
    X_train = np.concatenate(X_train_list, axis=0)
    y10_train = np.concatenate(y10_train_list, axis=0)
    y3_train = np.concatenate(y3_train_list, axis=0)
    del X_train_list, y10_train_list, y3_train_list
    gc.collect()

    X_val_list, y10_val_list, y3_val_list = [], [], []
    for s in val_sub_ids:
        epochs = load_one_subject(s, DATASET_PATH, skipped)
        if epochs is None:
            continue
        X, y10, y3 = epochs_to_arrays(epochs)
        del epochs
        gc.collect()
        if USE_EA:
            X = euclidean_align(X)
        X_val_list.append(X)
        y10_val_list.append(y10)
        y3_val_list.append(y3)
    X_val = np.concatenate(X_val_list, axis=0)
    y10_val = np.concatenate(y10_val_list, axis=0)
    y3_val = np.concatenate(y3_val_list, axis=0)
    del X_val_list, y10_val_list, y3_val_list
    gc.collect()

    epochs_test = load_one_subject(test_sub_id, DATASET_PATH, skipped)
    if epochs_test is None:
        tqdm.write(f"Skipping fold: no data for test subject sub-{test_sub_id}")
        continue
    X_test, y10_test, y3_test = epochs_to_arrays(epochs_test)
    del epochs_test
    gc.collect()
    if USE_EA:
        X_test = euclidean_align(X_test)

    X_train_n, _ = zscore_fit_transform(X_train, X_train)
    _, X_val_n = zscore_fit_transform(X_train, X_val)
    _, X_test_n = zscore_fit_transform(X_train, X_test)

    X_train_t = torch.from_numpy(X_train_n).float().unsqueeze(1)
    y_train_t = torch.from_numpy(y10_train).long()
    X_val_t = torch.from_numpy(X_val_n).float().unsqueeze(1)
    y_val_t = torch.from_numpy(y10_val).long()
    X_test_t = torch.from_numpy(X_test_n).float().unsqueeze(1)
    y10_test_t = torch.from_numpy(y10_test).long()
    y3_test_t = torch.from_numpy(y3_test).long()

    classes, counts = torch.unique(y_train_t, return_counts=True)
    class_weight = 1.0 / counts.float()
    sample_weight = class_weight[y_train_t]
    sampler = WeightedRandomSampler(sample_weight, len(sample_weight))

    train_ds = TensorDataset(X_train_t, y_train_t)
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=sampler, num_workers=0)
    val_loader = DataLoader(TensorDataset(X_val_t, y_val_t), batch_size=BATCH_SIZE, shuffle=False)

    model = EEGTransformer(n_classes=N_CLASSES_10, n_channels=N_CHANNELS, n_times=EXPECTED_N_TIMES).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=MAX_EPOCHS - WARMUP_EPOCHS)
    criterion = nn.CrossEntropyLoss()

    best_val_loss = float("inf")
    best_epoch = 0
    patience_counter = 0
    best_state = None

    pbar = tqdm(range(MAX_EPOCHS), desc=f"Epoch (fold sub-{test_sub_id})")
    for epoch in pbar:
        if epoch < WARMUP_EPOCHS:
            for g in opt.param_groups:
                g['lr'] = LR * (epoch + 1) / WARMUP_EPOCHS
        else:
            sched.step()
        model.train()
        train_loss = 0.0
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            opt.zero_grad()
            logits = model(xb)
            loss = criterion(logits, yb)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            opt.step()
            train_loss += loss.item() * xb.size(0)
        train_loss /= len(X_train_t)

        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for xb, yb in val_loader:
                xb, yb = xb.to(device), yb.to(device)
                logits = model(xb)
                val_loss += criterion(logits, yb).item() * xb.size(0)
        val_loss /= len(X_val_t)
        pbar.set_postfix(train_loss=f"{train_loss:.2f}", val_loss=f"{val_loss:.2f}")

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_epoch = epoch
            patience_counter = 0
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        else:
            patience_counter += 1
        if patience_counter >= EARLY_STOP_PATIENCE:
            break

    model.load_state_dict(best_state)
    model.to(device)
    model.eval()
    with torch.no_grad():
        logits = model(X_test_t.to(device))
        pred_10 = logits.argmax(dim=1).cpu().numpy()
        pred_3 = (pred_10 // 3).clip(0, 2)

    acc_10 = (pred_10 == y10_test).mean() * 100
    acc_3 = (pred_3 == y3_test).mean() * 100
    fold_results.append((test_sub_id, acc_10, acc_3))
    all_y10_true.append(y10_test)
    all_y10_pred.append(pred_10)
    all_y3_true.append(y3_test)
    all_y3_pred.append(pred_3)

    mean_3_so_far = np.mean([r[2] for r in fold_results])
    verdict = "STRONG" if mean_3_so_far > 50 else "MODERATE" if mean_3_so_far > 40 else "WEAK" if mean_3_so_far > 33 else "NULL"
    tqdm.write(f"Fold sub-{test_sub_id} | 10-class: {acc_10:.1f}% | 3-class: {acc_3:.1f}% | Verdict so far: {verdict}")

    if acc_3 > best_fold_acc_3:
        best_fold_acc_3 = acc_3
        best_fold_sub_id = test_sub_id
        best_fold_state = best_state
        best_fold_model = EEGTransformer(n_classes=N_CLASSES_10, n_channels=N_CHANNELS, n_times=EXPECTED_N_TIMES)
        best_fold_model.load_state_dict(best_state)

    torch.save(best_state, os.path.join(OUTPUTS_PATH, 'models', f'fold_sub{test_sub_id}_best.pt'))

    del train_loader, val_loader, train_ds, X_train, X_val, X_test, X_train_n, X_val_n, X_test_n
    del X_train_t, X_val_t, X_test_t, y10_train, y3_train, y10_val, y3_val
    gc.collect()

if skipped:
    print("Skipped (first occurrence):", skipped[:10])
print("Training done.")

## 8. Results and figures

Builds confusion matrices (10-class and 3-class), per-subject accuracy bar chart, saves logs and summary to OUTPUTS_PATH, and optionally the attention topomap. Prints the full PHASE 4 RESULTS block.

In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix

all_y10_true = np.concatenate(all_y10_true)
all_y10_pred = np.concatenate(all_y10_pred)
all_y3_true = np.concatenate(all_y3_true)
all_y3_pred = np.concatenate(all_y3_pred)

cm10 = confusion_matrix(all_y10_true, all_y10_pred, labels=list(range(10)))
cm3 = confusion_matrix(all_y3_true, all_y3_pred, labels=[0, 1, 2])
super_names = ["LIVING", "GEOMETRIC", "OBJECT"]

fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(cm10, cmap="Blues")
ax.set_xticks(range(10))
ax.set_yticks(range(10))
ax.set_xticklabels(CLASS_NAMES_10, rotation=45, ha="right")
ax.set_yticklabels(CLASS_NAMES_10)
ax.set_xlabel("Predicted")
ax.set_ylabel("True")
plt.colorbar(im, ax=ax, label="Count")
for i in range(10):
    for j in range(10):
        ax.text(j, i, int(cm10[i, j]), ha="center", va="center", fontsize=8)
plt.title("LOSO Combined Confusion Matrix (10-class)")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUTS_PATH, 'figures', 'confusion_matrix_10class.png'), dpi=150, bbox_inches="tight")
plt.close()

fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(cm3, cmap="Blues")
ax.set_xticks(range(3))
ax.set_yticks(range(3))
ax.set_xticklabels(super_names)
ax.set_yticklabels(super_names)
ax.set_xlabel("Predicted")
ax.set_ylabel("True")
plt.colorbar(im, ax=ax, label="Count")
for i in range(3):
    for j in range(3):
        ax.text(j, i, int(cm3[i, j]), ha="center", va="center", fontsize=10)
plt.title("LOSO Combined Confusion Matrix (3-class)")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUTS_PATH, 'figures', 'confusion_matrix_3class.png'), dpi=150, bbox_inches="tight")
plt.close()

subs = [f"sub-{s}" for s in fold_test_ids]
accs_10 = [r[1] for r in fold_results]
accs_3 = [r[2] for r in fold_results]
colors = ["C1" if s == f"sub-{FLAG_SUBJECT}" else "C0" for s in subs]
fig, ax = plt.subplots(figsize=(12, 4))
x = np.arange(len(subs))
ax.bar(x - 0.2, accs_10, 0.4, label="10-class", color=colors, alpha=0.8)
ax.bar(x + 0.2, accs_3, 0.4, label="3-class", color=colors, alpha=0.5)
ax.axhline(CHANCE_10, color="gray", linestyle="--", label=f"Chance 10-class ({CHANCE_10}%)")
ax.axhline(CHANCE_3, color="gray", linestyle=":", label=f"Chance 3-class ({CHANCE_3:.1f}%)")
ax.set_xticks(x)
ax.set_xticklabels(subs, rotation=45, ha="right")
ax.set_ylabel("Accuracy (%)")
ax.legend()
ax.set_title("Per-subject LOSO accuracy (sub-19 in orange)")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUTS_PATH, 'figures', 'per_subject_accuracy.png'), dpi=150, bbox_inches="tight")
plt.close()

rows = []
for sub_id, a10, a3 in fold_results:
    rows.append({"Subject": f"sub-{sub_id}", "10-class Acc": f"{a10:.2f}", "3-class Acc": f"{a3:.2f}", "Chance(10)": f"{CHANCE_10}", "Chance(3)": f"{CHANCE_3:.1f}"})
df = pd.DataFrame(rows)
df.to_csv(os.path.join(OUTPUTS_PATH, 'logs', 'loso_results.txt'), sep="\t", index=False)

recall_10 = np.diag(cm10) / (cm10.sum(axis=1) + 1e-9)
best_class_idx = int(np.argmax(recall_10))
worst_class_idx = int(np.argmin(recall_10))
best_class_name = CLASS_NAMES_10[best_class_idx]
worst_class_name = CLASS_NAMES_10[worst_class_idx]
best_class_acc = recall_10[best_class_idx] * 100
worst_class_acc = recall_10[worst_class_idx] * 100
best_sub_idx = np.argmax([r[2] for r in fold_results])
worst_sub_idx = np.argmin([r[2] for r in fold_results])
best_sub_id = fold_results[best_sub_idx][0]
worst_sub_id = fold_results[worst_sub_idx][0]
best_sub_acc = fold_results[best_sub_idx][2]
worst_sub_acc = fold_results[worst_sub_idx][2]
mean_10 = np.mean([r[1] for r in fold_results])
std_10 = np.std([r[1] for r in fold_results])
mean_3 = np.mean([r[2] for r in fold_results])
std_3 = np.std([r[2] for r in fold_results])
beats_10 = mean_10 - CHANCE_10
beats_3 = mean_3 - CHANCE_3
if mean_3 > 50 and mean_10 > 20:
    verdict = "STRONG"
elif mean_3 > 40 or mean_10 > 15:
    verdict = "MODERATE"
elif mean_3 > 33:
    verdict = "WEAK"
else:
    verdict = "NULL"

summary = (
    f"=== PHASE 4 RESULTS ===\n"
    f"Mean 10-class LOSO accuracy: {mean_10:.2f}% +/- {std_10:.2f}% (chance: 10%)\n"
    f"Mean 3-class LOSO accuracy: {mean_3:.2f}% +/- {std_3:.2f}% (chance: 33%)\n"
    f"Best decoded category: {best_class_name} at {best_class_acc:.1f}%\n"
    f"Worst decoded category: {worst_class_name} at {worst_class_acc:.1f}%\n"
    f"Best subject (easiest to decode): sub-{best_sub_id} at {best_sub_acc:.1f}%\n"
    f"Worst subject (hardest to decode): sub-{worst_sub_id} at {worst_sub_acc:.1f}%\n"
    f"Beats chance by: {beats_10:.1f} percentage points (10-class)\n"
    f"Beats chance by: {beats_3:.1f} percentage points (3-class)\n"
    f"DISCOVERY VERDICT: {verdict}"
)
print(summary)
with open(os.path.join(OUTPUTS_PATH, 'logs', 'phase4_summary.txt'), 'w', encoding='utf-8') as f:
    f.write(summary.strip() + "\n")

if best_fold_model is not None and best_fold_sub_id is not None and best_fold_state is not None:
    best_fold_model.load_state_dict(best_fold_state)
    weights = get_spatial_weights_for_topomap(best_fold_model)
    if weights is not None and len(weights) == len(ch_names):
        from mne import create_info
        info_32 = create_info(ch_names[:32], 250, "eeg")
        try:
            montage = mne.channels.make_standard_montage("standard_1020")
            info_32.set_montage(montage, on_missing="ignore", verbose=False)
        except Exception:
            pass
        fig, ax = plt.subplots(figsize=(5, 4))
        mne.viz.plot_topomap(weights, info_32, axes=ax, show=False)
        ax.set_title(f"Channel importance (best fold: sub-{best_fold_sub_id})")
        plt.tight_layout()
        plt.savefig(os.path.join(OUTPUTS_PATH, 'figures', 'attention_topomap.png'), dpi=150, bbox_inches="tight")
        plt.close()

print("Results and figures saved to", OUTPUTS_PATH)

## 9. Download results

Zips `vibci_outputs/` to `/kaggle/working/vibci_outputs.zip`. Download it from the run's **Output** tab when the notebook finishes.

In [ ]:
import zipfile

zipfile_path = '/kaggle/working/vibci_outputs.zip'
with zipfile.ZipFile(zipfile_path, 'w', zipfile.ZIP_DEFLATED) as zipf:
    for root, dirs, filenames in os.walk(OUTPUTS_PATH):
        for filename in filenames:
            filepath = os.path.join(root, filename)
            arcname = os.path.relpath(filepath, OUTPUTS_PATH)
            zipf.write(filepath, arcname)
print('Saved', zipfile_path, '- download from the Output tab after the run.')